# Keras を使用した MNIST 画像分類

このノートブックでは、Keras を使用して MNIST データセットに対する畳み込みニューラルネットワーク（CNN）を実装する方法を学びます。

**学習目標**
1. 畳み込み演算の仕組みを理解する。
2. 画像分類のための CNN の構築方法を理解する。
3. CNN の構造を理解するために、CNN が学習した特徴量を可視化する。

## セットアップ

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import Markdown, display
from keras import layers, models

## 畳み込み演算の可視化

**畳み込み（convolution）**は、画像処理における基本的な数学的演算です。これは、**カーネル（kernel）**または**フィルター（filter）**と呼ばれる小さな数値の行列を入力画像に適用する処理です。カーネルは画像上をスライドしながら、対応するピクセルと要素ごとの乗算を行い、その結果を合計して単一の出力ピクセルを生成します。このプロセスが画像全体で繰り返され、エッジやテクスチャなどの特定の特徴を際立たせる**特徴マップ（feature map）**が作成されます。

このセクションでは、シンプルな例を使って実際の動作を確認します。標準的な MNIST 画像を使用し、よく知られている 2 つのフィルター（Sobel X カーネルと Sobel Y カーネル）を適用して、水平方向と垂直方向のエッジを検出します。


Sobel カーネルは、エッジに対応するピクセル強度の方向性の変化を際立たせるように設計されています。Sobel X カーネルは水平方向のエッジを検出し、Sobel Y カーネルは垂直方向のエッジを検出します。


これらのカーネルを画像に適用すると、出力された特徴マップには検出されたエッジが明確に示され、Sobel X カーネルによって水平方向のエッジが、Sobel Y カーネルによって垂直方向のエッジが強調されます。これにより、単純な畳み込み演算によって画像から有意な特徴を抽出できることが実証されます。


In [ ]:
# MNIST データセットの読み込み
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 単一の画像を選択
image = x_train[0].astype("float32")
plt.imshow(image)
plt.title("Original Image")
plt.show()

それでは、水平方向のエッジを検出するものと垂直方向のエッジを検出するものの 2 つの Sobel カーネルを定義しましょう。

In [ ]:
# fmt: off
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

sobel_y = np.array([
    [-1, -2, -1],
    [0, 0, 0],
    [1, 2, 1]
], dtype=np.float32)
# fmt: on

plt.figure(figsize=(5, 5))
plt.subplot(1, 2, 1, title="Sobel X Kernel")
plt.imshow(sobel_x, cmap="gray")
plt.subplot(1, 2, 2, title="Sobel Y Kernel")
plt.imshow(
    sobel_y,
    cmap="gray",
)
plt.show()

次に、`Conv2D` レイヤーを使用してこれらのカーネルを画像に適用します。カーネルごとに 1 つずつ、合計 2 つの独立したレイヤーを作成します。

In [ ]:
image_reshaped = image.reshape(1, 28, 28, 1)
sobel_x_reshaped = sobel_x.reshape(3, 3, 1, 1)
sobel_y_reshaped = sobel_y.reshape(3, 3, 1, 1)

# Conv2D レイヤーの作成
conv_x = layers.Conv2D(1, (3, 3), use_bias=False, padding="valid")
conv_y = layers.Conv2D(1, (3, 3), use_bias=False, padding="valid")

conv_x.build(input_shape=(1, 28, 28, 1))
conv_y.build(input_shape=(1, 28, 28, 1))

conv_x.set_weights([sobel_x_reshaped])
conv_y.set_weights([sobel_y_reshaped])

# 畳み込みの適用
output_x = conv_x(image_reshaped)
output_y = conv_y(image_reshaped)

最後に、結果を可視化してみましょう。特徴マップには、検出された水平方向および垂直方向のエッジが表示されます。

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(output_x.numpy().reshape(26, 26))
plt.title("Horizontal Edges")
plt.subplot(1, 2, 2)
plt.imshow(output_y.numpy().reshape(26, 26))
plt.title("Vertical Edges")
plt.show()

## 畳み込みニューラルネットワーク（CNN）の構築とトレーニング

**畳み込みニューラルネットワーク（CNN: Convolutional Neural Network）**は、画像の処理や分類に優れたディープラーニングモデルの一種です。入力データから関連する特徴を自動的に抽出することを学習する層で構成されています。重要なコンセプトは、ネットワークが特徴の階層（初期の層ではエッジのような単純な特徴、より深い層では形状やオブジェクトのような複雑な特徴）を学習することです。

### データの前処理

MNIST 画像を CNN に入力する前に、データを準備する必要があります。主な 2 つのステップは以下のとおりです。

1.  **スケーリング**: MNIST 画像のピクセル値は 0 から 255 の範囲です。これらの値を 255.0 で割ることで正規化し、0 から 1 の範囲にスケールします。これにより、モデルのトレーニングがより効率的になります。

2.  **One-Hot エンコーディング**: 画像のラベルは単一の整数（0〜9）です。これを **One-Hot エンコードされたベクトル**に変換します。例えば、ラベル `5` はベクトル `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]` になり、正しい数字に対応するインデックスのみに 1 が設定されます。これは、マルチクラス分類問題における一般的な手法です。

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

### モデルアーキテクチャ

この CNN は、いくつかの層で構成されるシーケンシャルモデルです。基本的な考え方は、最終的な分類を行う前に、画像から特徴を段階的に抽出して洗練させることです。

このアーキテクチャは以下で構成されています。
* **畳み込み層（`Conv2D`）**: これらの層は入力にフィルターを適用し、特徴の検出を学習します。より豊かな特徴の階層を学習するために、フィルター数を徐々に増やした 4 つの層（16、32、64）を使用します。モデルがより複雑な関係を学習できるように、非線形性を導入する `relu` 活性化関数が使用されます。
* **最大プーリング層（`MaxPooling2D`）**: これらの層は畳み込み層の後に続きます。小さなウィンドウ内の最大値を取得して特徴マップをダウンサンプリングし、空間的次元の削減、計算コストの低減、過学習の防止に役立ちます。
* **Flatten 層**: この層は、2 次元の特徴マップを単一の 1 次元ベクトルに変形します。これは、ネットワークの畳み込み部分を高密度の分類層（Dense 層）に接続するために必要なステップです。
* **Dense 層（全結合層）**: これらは標準的な全結合ニューラルネットワーク層です。64 個のニューロンを持つ最初の Dense 層は、Flatten 層からの特徴を処理します。10 個のニューロン（各数字に 1 つずつ）を持つ最後の Dense 層は、`softmax` 活性化関数を使用して確率分布を出力し、可能性のある各数字に対するモデルの確信度を示します。

以下のモデルサマリーには、ネットワーク内の各層の出力形状とパラメータ数が示されています。


In [ ]:
model = models.Sequential(
    [
        layers.Conv2D(
            16,
            (3, 3),
            activation="relu",
            padding="same",
            input_shape=(28, 28, 1),
            name="Conv2D_1",
        ),
        layers.MaxPooling2D((2, 2), name="MaxPooling2D_1"),
        layers.Conv2D(
            32, (3, 3), activation="relu", padding="same", name="Conv2D_2"
        ),
        layers.MaxPooling2D((2, 2), name="MaxPooling2D_2"),
        layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="Conv2D_3"
        ),
        layers.MaxPooling2D((2, 2), name="MaxPooling2D_3"),
        layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="Conv2D_4"
        ),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model.summary()

### モデルのトレーニング

それでは、MNIST データセットでモデルをトレーニングします。バッチサイズ 128 を使用し、10 エポックにわたってトレーニングします。エポックとは、トレーニングデータセット全体に対する完全な 1 回のパス（学習周回）です。バッチサイズは、ネットワークを一度に伝播するサンプルの数です。

In [ ]:
history = model.fit(
    x_train, y_train, epochs=10, batch_size=128, validation_split=0.2
)

### モデルの評価

モデルのトレーニングが完了したので、テストデータでのパフォーマンスを評価する必要があります。モデルの評価には正解率（accuracy）の指標を使用します。正解率とは、正しく分類された画像の割合です。

### モデルの評価

トレーニング後、モデルがこれまでに見たことのない別のテストデータセットでモデルのパフォーマンスを評価します。正解率の指標は、モデルが正しく分類したテスト画像の割合を示します。

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)

### 失敗事例の確認

高い精度を誇るモデルであっても、完璧ではありません。誤分類された画像を分析することで、モデルの限界に関する洞察を得ることができます。これらのエラーの一部は、人間と AI の両方にとって解釈が難しい、曖昧に書かれた数字や乱雑に書かれた数字が原因である可能性があります。

In [ ]:
predictions = model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(y_test, axis=1)

misclassified_indices = np.where(predicted_labels != true_labels)[0]

plt.figure(figsize=(10, 10))
for i, index in enumerate(misclassified_indices[:9]):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_test[index].reshape(28, 28))
    plt.title(
        f"Predicted: {predicted_labels[index]}, True: {true_labels[index]}"
    )
    plt.axis("off")
plt.show()

## CNN の分析

### 学習したカーネルの可視化

畳み込み層内のフィルターは、特定の特徴を識別するためにモデルが学習したものです。これらのフィルターの重みを可視化することで、モデルが各段階で何を探しているかを把握できます。

ここでは、最初の畳み込み層からの 16 個のフィルターを表示します。各正方形は 3x3 のフィルターを表しています。フィルターが、さまざまな方向のエッジや単純なパターンなどの基本的な特徴を検出することを学習したことがわかります。これらの基本的な特徴は、後続の層で学習されるより複雑な特徴の構成要素となります。


In [ ]:
first_conv_layer = model.layers[0]
weights = first_conv_layer.get_weights()[0]

plt.figure(figsize=(10, 6))
for i in range(weights.shape[-1]):
    plt.subplot(4, 8, i + 1)
    plt.title(f"Kernel {i + 1}", fontsize=8)
    plt.imshow(weights[:, :, 0, i])
    plt.axis("off")
plt.show()

### 特徴マップの可視化

データが CNN を通過するにつれて、各層は入力を特徴マップに変換します。これらのマップは、その層のフィルターが検出することを学習した特徴の存在を強調します。異なる層のこれらのマップを可視化することで、ネットワークが入力画像のより抽象的な表現を段階的に構築していく様子を理解できます。

- **第 1 畳み込み層**: この層からの特徴マップは、学習されたカーネルによって検出された、エッジやコーナーなどの単純な特徴の存在を示します。
- **第 2 畳み込み層**: これらのマップは、第 1 層の出力にフィルターを適用した結果です。前の層で検出された単純な特徴の組み合わせである、より複雑な特徴を示しています。これは、CNN における特徴学習の階層的な性質を示しています。
- **第 3 および第 4 畳み込み層**: より深層に進むにつれて、特徴マップはより抽象的になり、人間の目には直接解釈しにくくなります。これらは、最終的な分類タスクに不可欠な、より高水準の特徴やパターンを表しています。

In [ ]:
def visualize_feature_map(activation):
    num_channel = activation.shape[-1]
    plt.figure(figsize=(10, num_channel // 8 * 1.5))
    for i in range(num_channel):
        plt.subplot(num_channel // 8, 8, i + 1)
        plt.title(f"Channel {i + 1}", fontsize=8)
        plt.imshow(activation[0, :, :, i], vmax=np.max(activation))
        plt.axis("off")
    plt.show()

In [ ]:
TEST_IMG_INDEX = 0  # 別の画像を確認するにはこのインデックスを変更します
CONV_ONLY = True  # MaxPooling も可視化する場合は False

layers = model.layers[0:7:2] if CONV_ONLY else model.layers[:7]
outputs = [layer.output for layer in layers]
activation_model = models.Model(inputs=model.inputs, outputs=outputs)

test_image = x_test[TEST_IMG_INDEX].reshape(1, 28, 28, 1)
activations = activation_model.predict(test_image)

display(Markdown(f"## Input Image, shape: {x_test[TEST_IMG_INDEX].shape}"))
plt.figure(figsize=(2, 2))
plt.imshow(x_test[TEST_IMG_INDEX])
plt.axis("off")
plt.show()

for i, activation in enumerate(activations):
    display(
        Markdown(f"## {layers[i].name}, output shape: {activation.shape[1:]}")
    )
    visualize_feature_map(activation)

この可視化は、CNN が生の画像ピクセルを、クラス情報に関する高度に抽象化されたシグナルを含む特徴セットへとどのように変換し、正確な分類を可能にしているかを示しています。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.